# 15-bob. 15-BOB. ETIKA, MAXFIYLIK VA HUQUQIY ASOSLAR
## Amaliy misol: taxalluslashning to'g'ri va noto'g'ri usuli

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/15-bob/15-2_taxalluslashning-togri-va-notogri-usuli.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 15-bob.*

Ko'pchilik dasturchilar shaxsiy ma'lumotni "yashirish" uchun uning xesh qiymatini saqlash yetarli deb hisoblaydi. Quyidagi dastur bu usulning xavfli ekanini ko'rsatadi. Xavfsizlik jurnalida telefon raqami o'rniga uning SHA-256 xeshi saqlangan. Buzg'unchi O'zbekistondagi telefon raqamlari fazosi juda kichik ekanini biladi va barcha mumkin bo'lgan raqamlarning xeshlarini hisoblab chiqadi. Namoyish uchun faqat bitta oraliqdagi 1 million raqam tekshiriladi. So'ngra to'g'ri usul, ya'ni maxfiy kalitli xesh (HMAC) ko'rsatiladi. Misoldagi raqam xayoliy. Dastur amaliy mashg'ulotning A15 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import hashlib
import hmac
import secrets
import time

# Jurnaldagi telefon raqamini "maxfiylashtirish" uchun oddiy SHA-256 xeshi ishlatilgan
raqam = "+998901234567"
oddiy_xesh = hashlib.sha256(raqam.encode()).hexdigest()
print("Jurnaldagi yozuv:", oddiy_xesh[:32], "...")

# Buzg'unchi raqamlar fazosi kichikligini biladi: +998 90 XXX XX XX (10 million variant).
# Namoyish uchun faqat +998 90 1XX XX XX oralig'ini (1 million variant) tekshiramiz.
boshlanish = time.time()
for i in range(1_000_000):
    nomzod = f"+998901{i:06d}"
    if hashlib.sha256(nomzod.encode()).hexdigest() == oddiy_xesh:
        print(f"Raqam tiklandi: {nomzod} ({time.time() - boshlanish:.1f} soniyada, {i + 1} urinish)")
        break

# To'g'ri usul: maxfiy kalitli xesh (HMAC). Kalit jurnaldan alohida, xavfsiz joyda saqlanadi
kalit = secrets.token_bytes(32)
taxallus = hmac.new(kalit, raqam.encode(), hashlib.sha256).hexdigest()
print("\nHMAC taxallus:", taxallus[:32], "...")
print("Bir xil raqam -> bir xil taxallus:",
      taxallus == hmac.new(kalit, raqam.encode(), hashlib.sha256).hexdigest())
topildi = any(hashlib.sha256(f"+9989012345{i:02d}".encode()).hexdigest() == taxallus for i in range(100))
print("Kalitsiz oddiy xeshlar bilan tiklab bo'ladimi:", topildi)

Jurnaldagi yozuv: b0ff57edaf3b57a5e0923896c2aeb095 ...


Raqam tiklandi: +998901234567 (0.3 soniyada, 234568 urinish)

HMAC taxallus: 06c1ddfc717df27e0aa70d1f7798c9af ...
Bir xil raqam -> bir xil taxallus: True
Kalitsiz oddiy xeshlar bilan tiklab bo'ladimi: False


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Jurnaldagi yozuv: b0ff57edaf3b57a5e0923896c2aeb095 ...
Raqam tiklandi: +998901234567 (0.2 soniyada, 234568 urinish)

HMAC taxallus: 2e1a0b174e3740c0464fae3d743499b1 ...
Bir xil raqam -> bir xil taxallus: True
Kalitsiz oddiy xeshlar bilan tiklab bo'ladimi: False
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 15-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.